In [ ]:
import os
import torch
from ultralytics import YOLO

In [ ]:
PROJECT_ROOT = os.getcwd() 

# הנתיב לקובץ ה-YAML המגדיר את הדאטה
DATA_YAML_PATH = os.path.join(PROJECT_ROOT, 'dataset', 'data.yaml')

# נתיב לתיקיית הטסט (לשימוש בסוף האימון לאבלואציה)
# וודא שהנתיב הזה נכון!
TEST_IMAGES_DIR = os.path.join(PROJECT_ROOT, 'dataset', 'images', 'test')

# הגדרות אימון (Hyperparameters)
MODEL_NAME = 'kitti_final_run'  
PRETRAINED_MODEL = 'yolov8n.pt' 
EPOCHS = 100                    
PATIENCE = 15                   
BATCH_SIZE = 16                 
IMG_SIZE = 640                  
WORKERS = 8      

In [ ]:
def check_system():
    print(f"{'-'*30}\nSystem Check:\n{'-'*30}")
    if torch.cuda.is_available():
        print(f"✅ GPU Detected: {torch.cuda.get_device_name(0)}")
        print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
    else:
        print("⚠️ Warning: No GPU detected. Training will be slow!")
    
    if not os.path.exists(DATA_YAML_PATH):
        raise FileNotFoundError(f"❌ Critical Error: Data YAML not found at {DATA_YAML_PATH}")
    else:
        print(f"✅ Data config found at: {DATA_YAML_PATH}")

In [ ]:
def train_model():
    
    print(f"\n🚀 Loading model: {PRETRAINED_MODEL}...")
    model = YOLO(PRETRAINED_MODEL)

    
    print(f"\n🏋️ Starting training for {MODEL_NAME}...")
    results = model.train(
        data=DATA_YAML_PATH,
        epochs=EPOCHS,
        patience=PATIENCE,       
        batch=BATCH_SIZE,
        imgsz=IMG_SIZE,
        workers=WORKERS,
        name=MODEL_NAME,         
        exist_ok=True,           
        pretrained=True,
        optimizer='auto',
        verbose=True,
        plots=True,              
        save=True,               
        device=0 if torch.cuda.is_available() else 'cpu'
    )
    
    return model

In [ ]:
def evaluate_on_test(model):
    print(f"\n{'-'*30}\n🧪 Running Evaluation on TEST Set\n{'-'*30}")
    
 
    try:
        metrics = model.val(split='test', name=f"{MODEL_NAME}_test_eval")
        print(f"✅ Test evaluation complete. mAP50-95: {metrics.box.map:.3f}")
    except Exception as e:
        print(f"⚠️ Could not run automatic test split validation (maybe 'test' path is missing in yaml?).")
        print(f"   Error: {e}")
        print("   Attempting manual prediction on test folder...")
        
        if os.path.exists(TEST_IMAGES_DIR):
            # הרצת חיזוי ושמירת התוצאות
            results = model.predict(
                source=TEST_IMAGES_DIR,
                save=True,
                name=f"{MODEL_NAME}_test_preds",
                conf=0.25 
            )
            print(f"✅ Predictions saved to runs/detect/{MODEL_NAME}_test_preds")
        else:
            print(f"❌ Test directory not found at {TEST_IMAGES_DIR}")

In [ ]:
if __name__ == '__main__':
    try:
        check_system()
        
      
        trained_model = train_model()
        
        evaluate_on_test(trained_model)
        
        print("\n🎉 Process Completed Successfully!")
        
    except Exception as e:
        print(f"\n❌ An error occurred: {e}")